# Portfolio Part 3 - Analysis of Loan Approval Data (2024 S2)

Name : Prashant Khanal

Student ID : 48310700

## Task background
In this Portfolio task, you will work on a new dataset named 'Loan Approval' which is a modified version from a synthetic Dataset for Risk Assessment and Loan Approval Modeling (many thanks to LORENZO ZOPPELLETTO for the sharing of this dataset). This dataset comprises 20,000 records of personal and financial data, designed to facilitate the development of predictive models for risk assessment and loan approval. In this portfolio part, you are mainly required to train classification models to determine the outcome of loan approval, indicating whether an applicant is likely to be approved or denied for a loan.

The dataset includes diverse features such as demographic information, credit history, employment status, income levels, existing debt, and other relevant financial metrics, providing a comprehensive foundation for sophisticated data-driven analysis and decision-making.

The dataset includes the following columns:

|Column|Meaning|
|:-----|:-----|
|ApplicationDate| Loan application date|
|Age| Applicant's age|
|AnnualIncome| Yearly income|
|CreditScore| Creditworthiness score|
|EmploymentStatus| Job situation|
|EducationLevel| Highest education attained|
|Experience| Work experience|
|LoanAmount| Requested loan size|
|LoanDuration| Loan repayment period|
|MaritalStatus| Applicant's marital state|
|NumberOfDependents| Number of dependents|
|HomeOwnershipStatus| Homeownership type|
|MonthlyDebtPayments| Monthly debt obligations|
|CreditCardUtilizationRate| Credit card usage percentage|
|NumberOfOpenCreditLines| Active credit lines|
|NumberOfCreditInquiries| Credit checks count|
|DebtToIncomeRatio| Debt to income proportion|
|BankruptcyHistory| Bankruptcy records|
|LoanPurpose| Reason for loan|
|PreviousLoanDefaults| Prior loan defaults|
|PaymentHistory| Past payment behavior|
|LengthOfCreditHistory| Credit history duration|
|SavingsAccountBalance| Savings account amount|
|CheckingAccountBalance| Checking account funds|
|TotalAssets| Total owned assets|
|TotalLiabilities| Total owed debts|
|MonthlyIncome| Income per month|
|UtilityBillsPaymentHistory| Utility payment record|
|JobTenure| Job duration|
|NetWorth| Total financial worth|
|BaseInterestRate| Starting interest rate|
|InterestRate| Applied interest rate|
|MonthlyLoanPayment| Monthly loan payment|
|TotalDebtToIncomeRatio| Total debt against income|
|LoanApproved| Loan approval status|
|RiskScore| Risk assessment score|

## Task Description
Your high level goal in this notebook is to build and evaluate predictive models for 'loan approval' from other available features. More specifically, you need to complete the following major tasks:

1. Clean and preprocess the dataset for the downstream data analysis tasks.

2. Build and evaluate logistic regression models with this datasets.

3. Build and evaluate KNN models with this datasets.

Note 1: While the main steps of each task have been given with the requirements, you should learn how to properly organise and comment your notebook by yourself to ensure that your notebook file is professional and readable.

Note 2: You will be evaluated on the accuracy of the model, the process that you produce the results,  and your clear description and justification of your implementation. So, try your best to comment your source code to showing your understanding and critical thinking.


In [124]:
#Importing essential Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,  confusion_matrix, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import RFE

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

## Load the dataset and show the basic information

In [75]:
loan_data = pd.read_csv('loan_approval.csv')
loan_data.head()

,ApplicationDate,Age,AnnualIncome,CreditScore,EmploymentStatus,EducationLevel,Experience,LoanAmount,LoanDuration,MaritalStatus,...,MonthlyIncome,UtilityBillsPaymentHistory,JobTenure,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,LoanApproved,RiskScore
0,2018-01-01,45.0,39948,617,Employed,Master,22,13152,48,Married,...,3329.000000,0.724972,11,126928,0.199652,0.227590,419.805992,0.181077,0,NaN
1,2018-01-02,38.0,39709,628,Employed,Associate,15,26045,48,Single,...,3309.083333,0.935132,3,43609,0.207045,0.201077,794.054238,0.389852,0,52.0
2,2018-01-03,47.0,40724,570,Employed,Bachelor,26,17627,36,Married,...,3393.666667,0.872241,6,5205,0.217627,0.212548,666.406688,0.462157,0,NaN
3,2018-01-04,58.0,69084,545,Employed,High School,34,37898,96,Single,...,5757.000000,0.896155,5,99452,0.300398,0.300911,1047.506980,0.313098,0,NaN
4,2018-01-05,37.0,103264,594,Employed,Associate,17,9184,36,Married,...,8605.333333,0.941369,5,227019,0.197184,0.175990,330.179140,0.070210,1,NaN


In [77]:
loan_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ApplicationDate             20000 non-null  object 
 1   Age                         19900 non-null  float64
 2   AnnualIncome                20000 non-null  int64  
 3   CreditScore                 20000 non-null  int64  
 4   EmploymentStatus            20000 non-null  object 
 5   EducationLevel              20000 non-null  object 
 6   Experience                  20000 non-null  int64  
 7   LoanAmount                  20000 non-null  int64  
 8   LoanDuration                20000 non-null  int64  
 9   MaritalStatus               19900 non-null  object 
 10  NumberOfDependents          20000 non-null  int64  
 11  HomeOwnershipStatus         20000 non-null  object 
 12  MonthlyDebtPayments         20000 non-null  int64  
 13  CreditCardUtilizationRate   200

In [79]:
loan_data.describe(include="all")

,ApplicationDate,Age,AnnualIncome,CreditScore,EmploymentStatus,EducationLevel,Experience,LoanAmount,LoanDuration,MaritalStatus,...,MonthlyIncome,UtilityBillsPaymentHistory,JobTenure,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,LoanApproved,RiskScore
count,20000,19900.000000,20000.000000,20000.000000,20000,20000,20000.000000,20000.000000,20000.000000,19900,...,20000.000000,20000.000000,20000.000000,2.000000e+04,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,1000.000000
unique,20000,NaN,NaN,NaN,3,5,NaN,NaN,NaN,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2018-01-01,NaN,NaN,NaN,Employed,Bachelor,NaN,NaN,NaN,Married,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,17036,6054,NaN,NaN,NaN,9999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,39.751759,59161.473550,571.612400,NaN,NaN,17.522750,24882.867800,54.057000,NaN,...,4891.715521,0.799918,5.002650,7.229432e+04,0.239124,0.239110,911.607052,0.402182,0.239000,50.687600
std,NaN,11.630809,40350.845168,50.997358,NaN,NaN,11.316836,13427.421217,24.664857,NaN,...,3296.771598,0.120665,2.236804,1.179200e+05,0.035509,0.042205,674.583473,0.338924,0.426483,7.881033
min,NaN,18.000000,15000.000000,343.000000,NaN,NaN,0.000000,3674.000000,12.000000,NaN,...,1250.000000,0.259203,0.000000,1.000000e+03,0.130101,0.113310,97.030193,0.016043,0.000000,30.400000
25%,NaN,31.750000,31679.000000,540.000000,NaN,NaN,9.000000,15575.000000,36.000000,NaN,...,2629.583333,0.727379,3.000000,8.734750e+03,0.213889,0.209142,493.763700,0.179693,0.000000,46.000000
50%,NaN,40.000000,48566.000000,578.000000,NaN,NaN,17.000000,21914.500000,48.000000,NaN,...,4034.750000,0.820962,5.000000,3.285550e+04,0.236157,0.235390,728.511452,0.302711,0.000000,52.000000
75%,NaN,48.000000,74391.000000,609.000000,NaN,NaN,25.000000,30835.000000,72.000000,NaN,...,6163.000000,0.892333,6.000000,8.882550e+04,0.261533,0.265532,1112.770759,0.509214,0.000000,56.000000


In [81]:
#Checking for Null values in columns
loan_data.isnull().sum()

ApplicationDate                   0
Age                             100
AnnualIncome                      0
CreditScore                       0
EmploymentStatus                  0
EducationLevel                    0
Experience                        0
LoanAmount                        0
LoanDuration                      0
MaritalStatus                   100
NumberOfDependents                0
HomeOwnershipStatus               0
MonthlyDebtPayments               0
CreditCardUtilizationRate         0
NumberOfOpenCreditLines           0
NumberOfCreditInquiries           0
DebtToIncomeRatio                 0
BankruptcyHistory                 0
LoanPurpose                       0
PreviousLoanDefaults              0
PaymentHistory                    0
LengthOfCreditHistory             0
SavingsAccountBalance             0
CheckingAccountBalance            0
TotalAssets                       0
TotalLiabilities                  0
MonthlyIncome                     0
UtilityBillsPaymentHistory  

In [92]:
df = loan_data.copy()

## Task 1: Clean the datasets (10 marks)

In [98]:
# Step 1: Convert 'ApplicationDate' to datetime format
df['ApplicationDate'] = pd.to_datetime(df['ApplicationDate'], errors='coerce')

# Step 2: Handle missing values
# Fill missing 'Age' with the median value
df['Age'].fillna(df['Age'].median(), inplace=True)

# Fill missing 'MaritalStatus' with the most common value (mode)
df['MaritalStatus'].fillna(df['MaritalStatus'].mode()[0], inplace=True)

# Fill missing 'RiskScore' with the median value
df['RiskScore'].fillna(df['RiskScore'].median(), inplace=True)

# Step 3: Remove duplicate rows if any
df.drop_duplicates(inplace=True)

# Optionally: check the final result
print(df.info())
print(df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   ApplicationDate             20000 non-null  datetime64[ns]
 1   Age                         20000 non-null  float64       
 2   AnnualIncome                20000 non-null  int64         
 3   CreditScore                 20000 non-null  int64         
 4   EmploymentStatus            20000 non-null  object        
 5   EducationLevel              20000 non-null  object        
 6   Experience                  20000 non-null  int64         
 7   LoanAmount                  20000 non-null  int64         
 8   LoanDuration                20000 non-null  int64         
 9   MaritalStatus               20000 non-null  object        
 10  NumberOfDependents          20000 non-null  int64         
 11  HomeOwnershipStatus         20000 non-null  object    

C:\Users\Prash\AppData\Local\Temp\ipykernel_11916\289614613.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\Prash\AppData\Local\Temp\ipykernel_11916\289614613.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

### Step 1.1 Handle the missing values with follwoing rules (5 marks)
1. If over 50% of the values of a column, the column should be removed from the data frame; 
2. For a categorical column, if a row contains a missing value, you need to delete the whole row; 
3. For a numerical column, if a row contains a missing value, you need to perform a missing value imputation with the average value of the column.

In [100]:
# Step 1: Removing columns with over 50% missing values
threshold = len(df) * 0.5
df.dropna(thresh=threshold, axis=1, inplace=True)

# Step 2: Removing rows with missing values in categorical columns
categorical_columns = df.select_dtypes(include=['object']).columns
df.dropna(subset=categorical_columns, inplace=True)

# Step 3: Imputing missing values in numerical columns with the mean
numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns
for column in numerical_columns:
    if df[column].isnull().sum() > 0:
        df[column].fillna(df[column].mean(), inplace=True)

# Checking the cleaned dataset for any remaining missing values
missing_values_after_cleaning = df.isnull().sum()

# Verifying the cleaning
print(df.info())
print(df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   ApplicationDate             20000 non-null  datetime64[ns]
 1   Age                         20000 non-null  float64       
 2   AnnualIncome                20000 non-null  int64         
 3   CreditScore                 20000 non-null  int64         
 4   EmploymentStatus            20000 non-null  object        
 5   EducationLevel              20000 non-null  object        
 6   Experience                  20000 non-null  int64         
 7   LoanAmount                  20000 non-null  int64         
 8   LoanDuration                20000 non-null  int64         
 9   MaritalStatus               20000 non-null  object        
 10  NumberOfDependents          20000 non-null  int64         
 11  HomeOwnershipStatus         20000 non-null  object    

### Step 1.2 Handle categorical attributes (5 marks)
1. If all the categorical values of a column are unique, this column does not provide any statistical informaiton and should be deleted.
2. Use one hot encoding to convert the categorical values into numerical ones.

In [102]:
# Step 1 
# Identifying categorical columns
categorical_columns = df.select_dtypes(include=['object']).columns

# Removing columns where all values are unique
for col in categorical_columns:
    if df[col].nunique() == len(df):
        df.drop(columns=[col], inplace=True)


In [104]:
# Step 2 
# Applying one-hot encoding to the remaining categorical columns
df = pd.get_dummies(df, drop_first=True)


In [106]:
#Verifying the changes
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 47 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   ApplicationDate                 20000 non-null  datetime64[ns]
 1   Age                             20000 non-null  float64       
 2   AnnualIncome                    20000 non-null  int64         
 3   CreditScore                     20000 non-null  int64         
 4   Experience                      20000 non-null  int64         
 5   LoanAmount                      20000 non-null  int64         
 6   LoanDuration                    20000 non-null  int64         
 7   NumberOfDependents              20000 non-null  int64         
 8   MonthlyDebtPayments             20000 non-null  int64         
 9   CreditCardUtilizationRate       20000 non-null  float64       
 10  NumberOfOpenCreditLines         20000 non-null  int64         
 11  Nu

## Task 2: Build a logistic regression classification model (25 marks)

In [110]:
# Step 1: Ensuring that only numeric columns are used in X
X = df.select_dtypes(include=['float64', 'int64']) 
y = df['LoanApproved']  # Target variable

# Step 2: Splitting the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Scaling the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 4: Training a logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Step 5: Making predictions
y_pred = model.predict(X_test)

# Step 6: Evaluating the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

# Displaying Output of the results
print(f"Accuracy: {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)


Accuracy: 100.00%
Confusion Matrix:
[[2983    0]
 [   0 1017]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2983
           1       1.00      1.00      1.00      1017

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000



### Step 2.1 Specify the features and the label, and split the dataset into training data and testing data (5 marks)
1. The attirbute "LoanApproved" is the label, which is the prediction target. The remaining attributes are the features.
2. The ratio for splitting the dataset is 80% for training and 20% for testing. Note that you need to set the "random_state" parameter as your student ID to produce your personlised splitting. Failing to do so will lose marks.

In [112]:
# Step 1: Specifying the features and label
X = df.drop(columns=['LoanApproved']) 
y = df['LoanApproved'] 

# Step 2: Splitting the dataset (80% training, 20% testing), using my student ID for random_state
student_id = 48310700  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=48310700)

# Displaying Output of the shape of the resulting datasets
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Testing labels shape: {y_test.shape}")


Training features shape: (16000, 46)
Testing features shape: (4000, 46)
Training labels shape: (16000,)
Testing labels shape: (4000,)


### Step 2.2 Build a logistic regression model (10 marks)
1. Train a logistic regression model
2. Report two classification performance metrics (accuracy and f1-score) on the testing data
3. Also report the two metrics on the training data, and compare the results with that of the testing data. Make a justification on whether the model is overfitting based on the comparison.

In [116]:
# Step 2.2: Handle datetime columns
df['ApplicationYear'] = pd.to_datetime(df['ApplicationDate']).dt.year
df['ApplicationMonth'] = pd.to_datetime(df['ApplicationDate']).dt.month

# Drop the original datetime column
df.drop('ApplicationDate', axis=1, inplace=True)
    
# Separate features and target
X = df.drop('LoanApproved', axis=1)
y = df['LoanApproved']

# Ensure all columns in X are numeric
X = X.select_dtypes(include=[float, int, bool])

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the logistic regression model
logreg = LogisticRegression(solver = 'saga', max_iter=1000)
logreg.fit(X_train, y_train)

# Step 2.2.1: Report metrics on testing data
y_pred_test = logreg.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)

# Step 2.2.2: Report metrics on training data
y_pred_train = logreg.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred_train)
train_f1 = f1_score(y_train, y_pred_train)

# Print the results
print("Testing Accuracy: {:.2f}".format(test_accuracy))
print("Testing F1-Score: {:.2f}".format(test_f1))
print("Training Accuracy: {:.2f}".format(train_accuracy))
print("Training F1-Score: {:.2f}".format(train_f1))

# Compare training and testing metrics to assess overfitting
if train_accuracy > test_accuracy + 0.05 and train_f1 > test_f1 + 0.05:
    print("The model may be overfitting.")
else:
    print("The model is not significantly overfitting.")


Testing Accuracy: 0.87
Testing F1-Score: 0.73
Training Accuracy: 0.87
Training F1-Score: 0.72
The model is not significantly overfitting.


C:\Users\Prash\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Step 2.3 Perform the recursive feature elimination (RFE) technique to identify the effective features for building the model (10 marks)
1. Visulise the change of the two performance metrics with respect to the number of eliminated features using a line chart.
2. In terms of the visualisation result, select a good value for the number of eliminated features with considering both performance maximisation and feature minimisation (two competing goals). Run the RFE again with the chosen number of eliminated features to obtain the corresponding set of retained features.

In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Perform RFE for different numbers of retained features
for num_features in range(1, X_train.shape[1] + 1):
# Perform RFE again using scaled data
rfe = RFE(logreg, n_features_to_select=num_features)
rfe.fit(X_train_scaled, y_train)

    # Make predictions for both training and testing sets
    y_pred_train = rfe.predict(X_train)
    y_pred_test = rfe.predict(X_test)
    
    # Calculate and store accuracy and F1-score for training data
    accuracy_scores_train.append(accuracy_score(y_train, y_pred_train))
    f1_scores_train.append(f1_score(y_train, y_pred_train))
    
    # Calculate and store accuracy and F1-score for testing data
    accuracy_scores_test.append(accuracy_score(y_test, y_pred_test))
    f1_scores_test.append(f1_score(y_test, y_pred_test))

In [ ]:
# Visualization of the change in performance metrics with respect to the number of retained features
plt.figure(figsize=(12, 8))

In [ ]:
# Plot accuracy scores
plt.plot(range(1, X_train.shape[1] + 1), accuracy_scores_train, label='Training Accuracy', marker='o')
plt.plot(range(1, X_train.shape[1] + 1), accuracy_scores_test, label='Testing Accuracy', marker='o')

In [ ]:
# Plot F1-scores
plt.plot(range(1, X_train.shape[1] + 1), f1_scores_train, label='Training F1-Score', marker='x')
plt.plot(range(1, X_train.shape[1] + 1), f1_scores_test, label='Testing F1-Score', marker='x')

In [ ]:
# Add titles and labels
plt.title("Performance Metrics vs Number of Retained Features")
plt.xlabel("Number of Retained Features")
plt.ylabel("Performance Metric")
plt.legend()
plt.grid(True)
plt.show()

## Task 3: Build a KNN classification model (25 marks)

In [ ]:
# Step 1: Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Build the KNN classification model
k = 5  
knn = KNeighborsClassifier(n_neighbors=k)

# Train the model
knn.fit(X_train_scaled, y_train)

# Step 3: Evaluate the model
# Predict on training and testing sets
y_pred_train = knn.predict(X_train_scaled)
y_pred_test = knn.predict(X_test_scaled)

# Step 4: Report accuracy and F1-score for both training and testing sets
accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_test = accuracy_score(y_test, y_pred_test)
f1_train = f1_score(y_train, y_pred_train)
f1_test = f1_score(y_test, y_pred_test)

print(f"Training Accuracy: {accuracy_train:.4f}")
print(f"Testing Accuracy: {accuracy_test:.4f}")
print(f"Training F1-Score: {f1_train:.4f}")
print(f"Testing F1-Score: {f1_test:.4f}")

# Optional: Tune the number of neighbors and observe the performance
k_values = range(1, 21)
accuracy_scores = []
f1_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    # Predict on the test set
    y_pred_test = knn.predict(X_test_scaled)
    
    # Calculate accuracy and F1-score
    accuracy_scores.append(accuracy_score(y_test, y_pred_test))
    f1_scores.append(f1_score(y_test, y_pred_test))

# Plot accuracy and F1-score with different k values
plt.figure(figsize=(12, 6))
plt.plot(k_values, accuracy_scores, label='Accuracy', marker='o')
plt.plot(k_values, f1_scores, label='F1-Score', marker='x')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Performance Metric')
plt.title('KNN: Performance vs Number of Neighbors')
plt.legend()
plt.grid(True)
plt.show()


### Step 3.1 Build 1-NN classifier (5 marks)
1. Slect the features identifed in Step 2.3 for this task
2. Buid 1-NN classifier and report two classification performance metrics (accuracy and f1-score) on the testing data
3. Also report the two metrics on the training data, and compare the results with that of the testing data. Make a justification on whether the model is overfitting based on the comparison.

In [ ]:
# Step 1: Select the features identified in Step 2.3
# Assuming `X_train_selected` and `X_test_selected` are the datasets with selected features
# This should contain the same features used in the RFE process.

# Step 2: Standardize the selected features
scaler = StandardScaler()
X_train_selected_scaled = scaler.fit_transform(X_train_selected)
X_test_selected_scaled = scaler.transform(X_test_selected)

# Step 3: Build the 1-NN classifier
knn_1 = KNeighborsClassifier(n_neighbors=1)

# Train the model
knn_1.fit(X_train_selected_scaled, y_train)

# Step 4: Evaluate the model
# Predict on training and testing sets
y_pred_train = knn_1.predict(X_train_selected_scaled)
y_pred_test = knn_1.predict(X_test_selected_scaled)

# Calculate accuracy and F1-score for both sets
accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_test = accuracy_score(y_test, y_pred_test)
f1_train = f1_score(y_train, y_pred_train, average='weighted')
f1_test = f1_score(y_test, y_pred_test, average='weighted')

# Print results
print(f"Training Accuracy: {accuracy_train:.4f}")
print(f"Testing Accuracy: {accuracy_test:.4f}")
print(f"Training F1-Score: {f1_train:.4f}")
print(f"Testing F1-Score: {f1_test:.4f}")

# Step 5: Justification of Overfitting
if accuracy_train > accuracy_test:
    print("The model may be overfitting, as it performs better on training data than on testing data.")
elif accuracy_train == accuracy_test and f1_train == f1_test:
    print("The model appears to be well-generalized, performing similarly on both training and testing data.")
else:
    print("The model does not exhibit signs of overfitting, with good performance on both training and testing data.")


### Step 3.2 Use the grid search and cross validation techniques to study the performance change with respect to the hyperparameter K (10 marks)
1. User grid search to search K in the range (1, 30) both inclusive with 5-fold cross validation. The performance metric used for search is accuracy.
2. Visualise the performance change with respect to K using a line chart. Report the two performance metrics for the best case.

### Step 3.3 Study how the distance metrics affect the model performance (10 marks)
1. Change the distance metric parameter to 3 distance types: 'euclidean'(also l2), 'l1', and 'cosine', respectively, and visualise the model performance with these 3 distances, using a bar chart for both accuracy and f1 scores.
2. Compare the performance metrics, which is the best? Which is the worest?